# Chapter 8 - Learning Signal and Ignoring Noise: Regularization and Batching

## Summary
Deals with **overfitting**: a network memorizes training data instead of learning general patterns.
- **3-layer network on digits:** trains fast but test accuracy lags training accuracy -> overfitting.
- **Overfitting:** the model memorizes noise specific to the training set; training error drops while test error rises.
- **Early stopping:** stop training when the test/validation accuracy stops improving.
- **Dropout:** randomly switch off neurons (set to 0) during training; scale the rest by 1/(1-p). Acts like training many small networks and averaging them.
- **Batch gradient descent:** update weights using the average gradient of a mini-batch; faster (vectorized) and smoother.

## Theory
- **Regularization:** any technique that discourages memorization (early stopping, dropout, weight decay, more data).
- **Why overfitting happens:** a flexible model can find *coincidental* correlations in a small dataset.
- **Dropout intuition:** neurons can't rely on specific partners, so they must learn robust, independent features; the ensemble effect reduces variance.
- **Inverted dropout:** multiply surviving activations by `1/(1-p)` so the expected value is unchanged and no scaling is needed at test time.
- **Mini-batching:** the gradient averaged over `B` samples is a less noisy estimate; also uses fast matrix ops.

In [1]:
# If numpy / scikit-learn are missing in this kernel, uncomment:
# %pip install numpy scikit-learn
import numpy as np

In [2]:
# Dataset: scikit-learn's bundled 8x8 handwritten digits (offline stand-in for MNIST's 28x28).
# The code is identical in spirit; only image size differs. Swap in MNIST if you like.
from sklearn.datasets import load_digits
digits = load_digits()
X = digits.data / 16.0                      # pixels scaled to 0..1
y = digits.target
labels = np.eye(10)[y]                      # one-hot
n_train = 1000
images, train_labels = X[:n_train], labels[:n_train]
test_images, test_labels = X[n_train:], labels[n_train:]
print(images.shape, test_images.shape)

(1000, 64) (797, 64)


In [3]:
relu = lambda x: (x >= 0) * x
relu2deriv = lambda x: x >= 0

def train(images, labels, test_images, test_labels, hidden=40, alpha=0.005, iters=300,
          dropout=False, batch_size=1, seed=1, log_every=50):
    np.random.seed(seed)
    w01 = 0.2 * np.random.random((images.shape[1], hidden)) - 0.1
    w12 = 0.2 * np.random.random((hidden, 10)) - 0.1
    for j in range(iters):
        err = correct = 0
        for i in range(len(images) // batch_size):
            s, e = i * batch_size, (i + 1) * batch_size
            l0 = images[s:e]
            l1 = relu(l0.dot(w01))
            if dropout:
                mask = np.random.randint(2, size=l1.shape)
                l1 *= mask * 2
            l2 = l1.dot(w12)
            err += np.sum((labels[s:e] - l2) ** 2)
            correct += sum(np.argmax(l2[k]) == np.argmax(labels[s + k]) for k in range(batch_size))
            d2 = (labels[s:e] - l2) / batch_size
            d1 = d2.dot(w12.T) * relu2deriv(l1)
            if dropout: d1 *= mask
            w12 += alpha * l1.T.dot(d2)
            w01 += alpha * l0.T.dot(d1)
        if j % log_every == 0 or j == iters - 1:
            l1t = relu(test_images.dot(w01)); l2t = l1t.dot(w12)
            terr = np.sum((test_labels - l2t) ** 2)
            tcor = np.mean(np.argmax(l2t, 1) == np.argmax(test_labels, 1))
            print(f"I:{j:4d} Train-Err:{err/len(images):.3f} Train-Acc:{correct/len(images):.3f} "
                  f"Test-Err:{terr/len(test_images):.3f} Test-Acc:{tcor:.3f}")
    return w01, w12

### 1) Plain 3-layer network (watch it overfit)

In [4]:
train(images, train_labels, test_images, test_labels, hidden=100, alpha=0.005, iters=300);

I:   0 Train-Err:0.763 Train-Acc:0.560 Test-Err:0.615 Test-Acc:0.799
I:  50 Train-Err:0.168 Train-Acc:0.985 Test-Err:0.260 Test-Acc:0.917
I: 100 Train-Err:0.151 Train-Acc:0.985 Test-Err:0.251 Test-Acc:0.908
I: 150 Train-Err:0.147 Train-Acc:0.981 Test-Err:0.256 Test-Acc:0.898
I: 200 Train-Err:0.145 Train-Acc:0.984 Test-Err:0.260 Test-Acc:0.902
I: 250 Train-Err:0.144 Train-Acc:0.984 Test-Err:0.262 Test-Acc:0.901
I: 299 Train-Err:0.143 Train-Acc:0.984 Test-Err:0.261 Test-Acc:0.905


### 2) With dropout (50%)

In [5]:
train(images, train_labels, test_images, test_labels, hidden=100, alpha=0.005, iters=300, dropout=True);

I:   0 Train-Err:0.869 Train-Acc:0.310 Test-Err:0.721 Test-Acc:0.770
I:  50 Train-Err:0.396 Train-Acc:0.855 Test-Err:0.326 Test-Acc:0.898
I: 100 Train-Err:0.352 Train-Acc:0.888 Test-Err:0.285 Test-Acc:0.900
I: 150 Train-Err:0.324 Train-Acc:0.888 Test-Err:0.269 Test-Acc:0.898
I: 200 Train-Err:0.317 Train-Acc:0.901 Test-Err:0.264 Test-Acc:0.897
I: 250 Train-Err:0.298 Train-Acc:0.904 Test-Err:0.262 Test-Acc:0.898
I: 299 Train-Err:0.324 Train-Acc:0.914 Test-Err:0.281 Test-Acc:0.897


### 3) With mini-batches (size 100): same data, far fewer weight updates per epoch, faster per epoch

In [6]:
train(images, train_labels, test_images, test_labels, hidden=100, alpha=0.1, iters=300, dropout=True, batch_size=100);

I:   0 Train-Err:1.031 Train-Acc:0.103 Test-Err:0.882 Test-Acc:0.171
I:  50 Train-Err:0.441 Train-Acc:0.854 Test-Err:0.408 Test-Acc:0.880
I: 100 Train-Err:0.405 Train-Acc:0.875 Test-Err:0.348 Test-Acc:0.895
I: 150 Train-Err:0.402 Train-Acc:0.868 Test-Err:0.344 Test-Acc:0.903
I: 200 Train-Err:0.401 Train-Acc:0.866 Test-Err:0.339 Test-Acc:0.897
I: 250 Train-Err:0.380 Train-Acc:0.880 Test-Err:0.336 Test-Acc:0.893
I: 299 Train-Err:0.391 Train-Acc:0.864 Test-Err:0.331 Test-Acc:0.895


## Takeaways
Watch test accuracy, not training accuracy. Dropout and batching are cheap, effective ways to generalise better and train faster.